# Kaggle Run From Git

Thin Kaggle notebook: clone or pull the latest repo code, then run the Python entrypoint.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/Internship.git"
REPO_DIR = Path("/kaggle/working/Internship")
BRANCH = None

# --- Run settings ---
SMOKE = True            # True -> quick 1-epoch check before the real run
EPOCHS = 50
BATCH_SIZE = 16
MODELS = ["custom_cnn_v2"]

# --- Tier-1 training options (defaults reproduce the old behaviour) ---
OPTIMIZER = "adamw"              # adam | adamw
LR_SCHEDULER = "cosine_warmup"   # none | cosine | cosine_warmup
LABEL_SMOOTHING = 0.1
LOSS = "cb_focal"                # ce | focal | class_balanced | cb_focal
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0


In [ ]:
import subprocess

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "--all"], cwd=REPO_DIR, check=True)
    if BRANCH:
        subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    command = ["git", "clone"]
    if BRANCH:
        command.extend(["--branch", BRANCH])
    command.extend([REPO_URL, str(REPO_DIR)])
    subprocess.run(command, check=True)

print("Repository ready:", REPO_DIR)

In [ ]:
%cd /kaggle/working/Internship
import subprocess

epochs = 1 if SMOKE else EPOCHS
cmd = [
    "python", "scripts/train_kaggle.py",
    "--epochs", str(epochs),
    "--batch-size", str(BATCH_SIZE),
    "--models", *MODELS,
    "--optimizer", OPTIMIZER,
    "--lr-scheduler", LR_SCHEDULER,
    "--label-smoothing", str(LABEL_SMOOTHING),
    "--loss", LOSS,
    "--mixup-alpha", str(MIXUP_ALPHA),
    "--cutmix-alpha", str(CUTMIX_ALPHA),
]
if SMOKE:
    cmd.append("--no-save-every-epoch")
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path

output_root = Path("/kaggle/working/plant_training_outputs")
for path in sorted(output_root.rglob("*")):
    if path.is_file():
        print(path.relative_to(output_root))